# DLBCL Donor Pipeline Run

This notebook runs the refactored donor pipeline from configuration through folder-structure modeling, image processing, table building, and result inspection.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dlbcl_pipeline").exists():
    REPO_ROOT = Path.cwd() / "dlbcl-deep-learning"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
from dlbcl_pipeline.classification.classify_tcells import classify_tcells
from dlbcl_pipeline.config import build_local_pipeline_config
from dlbcl_pipeline.export import build_formatted_channels
from dlbcl_pipeline.measurements.aggregation import build_donor_table
from dlbcl_pipeline.model_folder_structure import build_donor_folder_structure
from dlbcl_pipeline.plotting.intensity_distributions import plot_donor_intensity_distributions
from dlbcl_pipeline.process_donor import process_donor


## Configuration

In [ ]:
BASE_PATH = Path("/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241")
PROJECT_DATA_ROOT = REPO_ROOT.parent

# One source of truth for the run selection.
# Use {} or None to process every sample/image.
SAMPLES_TO_PROCESS = {1: [1, 2, 3, 4, 5]}

EXPORT_CHANNELS = ("actin", "ccr7", "cd45ra")
CLEAN_FORMATTED_OUTPUT = True

ANNOUNCE_FILTERS = True
VERBOSE = False

config = build_local_pipeline_config(
    donor_dir=BASE_PATH,
    samples_to_process=SAMPLES_TO_PROCESS,
)

run_selection = {
    "donor_dir": str(config.donor_dir),
    "samples_to_process": config.samples_to_process,
    "images_to_process": config.images_to_process,
}

run_selection

## Model Folder Structure

In [ ]:
donor_folder_structure = build_donor_folder_structure(
    config,
    announce_filters=ANNOUNCE_FILTERS,
)

donor_folder_structure

In [ ]:
structure_rows = []
for sample in donor_folder_structure.samples:
    for image in sample.images:
        structure_rows.append({
            "sample": sample.name,
            "sample_number": sample.sample_number,
            "image": image.name,
            "image_number": image.image_number,
            "image_path": str(image.path),
            "channels": {key: path.name for key, path in image.channels.items()},
        })

structure_rows

## Run Processing

In [ ]:
# sometimes, this cell will fail if the kernel/python process is old. 
# to fix this, restart the kernel

donor_result = process_donor(
    config,
    donor_folder_structure=donor_folder_structure,
    verbose=VERBOSE,
)

{
    "success": donor_result.success,
    "error": donor_result.error,
    "total_images": donor_result.total_images,
    "total_processed": donor_result.total_processed,
    "total_failed": donor_result.total_failed,
}


## Result Summary

In [ ]:
if "donor_result" not in globals():
    raise RuntimeError("Run the 'Run Processing' cell first so donor_result is defined.")

summary = {
    "success": donor_result.success,
    "error": donor_result.error,
    "donor_dir": str(donor_result.donor_dir),
    "total_images": donor_result.total_images,
    "total_processed": donor_result.total_processed,
    "total_failed": donor_result.total_failed,
    "donor_table": str(donor_result.donor_table.output_path) if donor_result.donor_table and donor_result.donor_table.output_path else None,
}

summary

In [ ]:
if "donor_result" not in globals():
    raise RuntimeError("Run the 'Run Processing' cell first so donor_result is defined.")

image_result_rows = []
for image_result in donor_result.image_results:
    image_result_rows.append({
        "sample": image_result.sample_name,
        "image": image_result.image_name,
        "success": image_result.success,
        "error": image_result.error,
        "result_keys": sorted(image_result.results.keys()),
    })

image_result_rows

In [ ]:
if "image_result_rows" not in globals():
    raise RuntimeError("Run the image result summary cell first so image_result_rows is defined.")

failed_images = [row for row in image_result_rows if not row["success"]]
failed_images

## Rebuild Selected Donor CSV


In [ ]:
# This rebuilds sample-level and donor-level CSVs using the same selected samples/images from config.
donor_table_result = build_donor_table(
    config,
    verbose=VERBOSE,
)

if not donor_table_result.success:
    raise RuntimeError(donor_table_result.error)

{
    "output_path": str(donor_table_result.output_path),
    "rows": donor_table_result.rows,
    "columns": donor_table_result.columns,
}


## Plot Selected Intensity Distributions


In [ ]:
if "donor_table_result" not in globals():
    raise RuntimeError("Run the 'Rebuild Selected Donor CSV' cell first.")

from dlbcl_pipeline.plotting.intensity_distributions import plot_donor_intensity_distributions

plot_result = plot_donor_intensity_distributions(
    donor_dir=config.donor_dir,
    csv_file=Path(donor_table_result.output_path).name,
    output_file="selected_images_intensity_histograms.png",
    donor_label=config.donor_dir.name,
    verbose=VERBOSE,
)

if not plot_result["success"]:
    raise RuntimeError(plot_result["error"])

plot_result


## Classify Selected Donor CSV


In [ ]:
if "donor_table_result" not in globals():
    raise RuntimeError("Run the 'Rebuild Selected Donor CSV' cell first.")

from dlbcl_pipeline.classification.classify_tcells import classify_tcells

# Edit these thresholds after inspecting the intensity distribution plot.
CLASSIFICATION_THRESHOLDS = {
    "cd4": 260.0,
    "cd45ra": 120.0,
    "ccr7": 114.0,
    "cd19car": 0.0,
}

classification_result = classify_tcells(
    base_path=config.donor_dir,
    input_csv=Path(donor_table_result.output_path).name,
    output_csv=config.export.classified_csv,
    thresholds=CLASSIFICATION_THRESHOLDS,
    verbose=VERBOSE,
)

if not classification_result["success"]:
    raise RuntimeError(classification_result.get("error", "Classification failed"))

classification_result


## Export Formatted Channel Images


## Normalize Channel Images

In [ ]:
from dlbcl_pipeline.image_processing.normalize import normalize_donor_channels

normalization_results = normalize_donor_channels(
    donor_dir=config.donor_dir,
    channels=EXPORT_CHANNELS,
    verbose=VERBOSE,
)

{ch: {"success": r["success"], "num_processed": r.get("num_processed"), "error": r.get("error")}
 for ch, r in normalization_results.items()}

## Stack RGB Channels

In [ ]:
from dlbcl_pipeline.image_processing.rgb_stack import stack_rgb_channels

rgb_result = stack_rgb_channels(
    donor_dir=config.donor_dir,
    r_channel="ccr7",
    g_channel="actin",
    b_channel="cd45ra",
    verbose=VERBOSE,
)

if not rgb_result["success"]:
    raise RuntimeError(rgb_result["error"])

rgb_result

In [ ]:
if "classification_result" not in globals():
    raise RuntimeError("Run the 'Classify Selected Donor CSV' cell first.")

# Set this path after segmentation and manual review of bad cells.
CLEAN_CELL_LIST_PATH = PROJECT_DATA_ROOT / "clean_cell_list.csv"

if not CLEAN_CELL_LIST_PATH.exists():
    raise FileNotFoundError(f"Clean cell list not found: {CLEAN_CELL_LIST_PATH}")

try:
    donor_folder_path = config.donor_dir.relative_to(PROJECT_DATA_ROOT)
except ValueError:
    donor_folder_path = config.donor_dir

formatted_export_result = build_formatted_channels(
    root=PROJECT_DATA_ROOT,
    donor_folder_path=donor_folder_path,
    clean_csv=CLEAN_CELL_LIST_PATH,
    channels=EXPORT_CHANNELS,
    samples_to_process=config.samples_to_process,
    images_to_process=config.images_to_process,
    classified_csv=config.export.classified_csv,
    clean_output=CLEAN_FORMATTED_OUTPUT,
    verbose=VERBOSE,
)

formatted_export_result

In [ ]:
if "formatted_export_result" not in globals():
    raise RuntimeError("Run the formatted channel export cell first.")

formatted_counts = {}
missing_formatted_sources = []
for result in formatted_export_result.get("channel_results", []):
    output_dir = Path(result["output_dir"])
    channel = result["channel"]
    formatted_counts[channel] = {
        "output_dir": str(output_dir),
        "files": len(list(output_dir.glob("*.tif"))) if output_dir.exists() else 0,
        "copied": result["copied"],
        "missing": result["missing"],
    }
    for item in result.get("missing_entries", []):
        missing_formatted_sources.append({
            "channel": channel,
            "sample": item["sample"],
            "image": item["image"],
            "cell": item["cell"],
            "expected_source_dir": item["expected_source_dir"],
            "target_name": item["target_name"],
        })

{
    "counts": formatted_counts,
    "missing_sources": missing_formatted_sources,
}


## Inspect Channel Crop Artifacts


In [ ]:
if "donor_folder_structure" not in globals():
    raise RuntimeError("Run the 'Model Folder Structure' cell first so donor_folder_structure is defined.")

artifact_dirs = (
    "raw_actin",
    "raw_ccr7",
    "raw_cd45ra",
    "padded_cells",
    "padded_ccr7",
    "padded_cd45ra",
)

artifact_rows = []
for sample in donor_folder_structure.samples:
    for image in sample.images:
        row = {
            "sample": sample.name,
            "image": image.name,
        }
        for dirname in artifact_dirs:
            folder = image.path / dirname
            row[dirname] = len(list(folder.glob("*.tif"))) if folder.exists() else None
        artifact_rows.append(row)

artifact_rows
